In [11]:
import json
import csv

# Load multiple JSON objects from a file
with open("output/multi_run/NCT04898894_IC_run_1.json", "r") as f:
    data = json.load(f)

# Determine max number of quotes across all entries
max_quotes = 0
for entry in data:
    if "answer_facts" in entry and entry["answer_facts"]:
        max_quotes = max(max_quotes, len(entry["answer_facts"][0].get("substring_quote", [])))

# Prepare CSV
with open("output.csv", "w", newline="") as f:
    writer = csv.writer(f)

    # Header
    header = ["entity", "exist", "value", "fact"]
    for i in range(1, max_quotes + 1):
        header += [f"quote{i}", f"index{i}"]
    writer.writerow(header)

    for entry in data:
        for key in entry:
            if key.endswith("_exist"):
                entity = key.replace("_exist", "")
                exist = entry[key]
                value = entry.get(entity, "")

                # Check and extract facts
                if "answer_facts" in entry and entry["answer_facts"]:
                    fact_obj = entry["answer_facts"][0]
                    fact = fact_obj.get("fact", "")
                    quotes = fact_obj.get("substring_quote", [])
                    indices = fact_obj.get("substring_indices", [])
                else:
                    fact = ""
                    quotes = []
                    indices = []

                row = [entity, exist, value, fact]
                for q, idx in zip(quotes, indices):
                    row.extend([q, str(idx)])  # you can use `json.dumps(idx)` for JSON format

                # Pad to match header length
                while len(row) < len(header):
                    row.append("")
                writer.writerow(row)
